# Reproduction notebook: 16_official_patchtst_plus_frozen_retrieval_h96

This notebook is retained as an executable provenance record for the anonymous supplementary package. Saved outputs and internal development notes have been removed.


In [ ]:

from pathlib import Path
from types import SimpleNamespace
from contextlib import nullcontext

import gc
import importlib
import math
import random
import sys
import time
import warnings

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 380)

# ============================================================
# Frozen experiment specification
# ============================================================

DATASETS = ["ETTm1", "ETTh1"]   # run both without changing settings
H = 96

DIRECT_SEQ_LEN = 336
RET_SEQ_LEN = 96

SEED = 0
CROSSFIT_SEED = 1313

# Frozen retrieval settings from Exp13
TOP_K = 10
MEMORY_STRIDE = 24
OOF_ANCHOR_STRIDE = 4
FOLDS = [
    (0.55, 0.70),
    (0.70, 0.85),
    (0.85, 1.00),
]

# Frozen predictive representation
REP_PATCH_LEN = 16
REP_PATCH_STRIDE = 16
REP_D_MODEL = 64
REP_N_HEADS = 4
REP_LAYERS = 2
REP_D_FF = 128
REP_DIM = 64
REP_DROPOUT = 0.1
REP_NUM_PATCHES = (
    1
    + (
        RET_SEQ_LEN
        - REP_PATCH_LEN
    )
    // REP_PATCH_STRIDE
)

# Frozen gate
GATE_DIM = 26
GATE_LR = 1e-3
GATE_WD = 1e-4
GATE_BATCH = 8192
GATE_MAX_EPOCHS = 50
GATE_PATIENCE = 7

ALPHA_GRID = np.round(
    np.arange(
        0.0,
        1.0001,
        0.1,
    ),
    10,
)

LAMBDA_GRID = np.array(
    [
        0.0,
        0.25,
        0.50,
        0.75,
        1.00,
    ],
    dtype=np.float32,
)

# Evaluation / memory
PAIR_BATCH = 256
FULL_ANCHOR_BATCH = 8
EPS = 1e-8

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

RETRIEVER_USE_AMP = torch.cuda.is_available()

# ============================================================
# Existing experiment roots
# ============================================================

OFFICIAL_ROOT = Path(
    "/data/dataset/strong_forecaster/"
    "official_patchtst_h96_reproduction"
)

FROZEN_RET_ROOT = Path(
    "/data/dataset/strong_forecaster/"
    "multidataset_crossfit_screening"
)

# New experiment root
ROOT = Path(
    "/data/dataset/strong_forecaster/"
    "official_patchtst_plus_frozen_retrieval_h96"
)

DIRS = {
    "fold_direct":
        ROOT / "fold_direct",
    "oof":
        ROOT / "oof",
    "gate":
        ROOT / "gate",
    "history":
        ROOT / "history",
    "paired":
        ROOT / "paired_test",
}

for p in DIRS.values():
    p.mkdir(
        parents=True,
        exist_ok=True,
    )

RESUME = True
FORCE = False

print("Device:", DEVICE)
print("Datasets:", DATASETS)
print("Horizon:", H)
print("Official direct root:", OFFICIAL_ROOT)
print("Frozen retriever root:", FROZEN_RET_ROOT)
print("Output:", ROOT)


In [ ]:

OFFICIAL_REPO_CANDIDATES = [
    Path(
        "/code/stock_regime_retrieval/"
        "strong_forecaster/PatchTST_official"
    ),
    Path("/code/PatchTST_official"),
    Path("/data/PatchTST_official"),
    Path("/data/PatchTST"),
]

OFFICIAL_REPO = next(
    (
        p
        for p in OFFICIAL_REPO_CANDIDATES
        if (
            p
            / "PatchTST_supervised"
            / "models"
            / "PatchTST.py"
        ).exists()
    ),
    None,
)

if OFFICIAL_REPO is None:
    raise FileNotFoundError(
        "Official PatchTST repository was not found. "
        "Run Experiment 15 first."
    )

SUPERVISED_ROOT = (
    OFFICIAL_REPO
    / "PatchTST_supervised"
)

# Avoid mixing Time-Series-Library's `models` / `layers`
# with official PatchTST modules.
for module_name in list(
    sys.modules.keys()
):
    if (
        module_name == "models"
        or module_name.startswith(
            "models."
        )
        or module_name == "layers"
        or module_name.startswith(
            "layers."
        )
    ):
        del sys.modules[
            module_name
        ]

if str(
    SUPERVISED_ROOT
) in sys.path:
    sys.path.remove(
        str(
            SUPERVISED_ROOT
        )
    )

sys.path.insert(
    0,
    str(
        SUPERVISED_ROOT
    ),
)

patchtst_module = importlib.import_module(
    "models.PatchTST"
)

OfficialPatchTST = (
    patchtst_module.Model
)

actual_model_file = Path(
    patchtst_module.__file__
).resolve()

expected_model_file = (
    SUPERVISED_ROOT
    / "models"
    / "PatchTST.py"
).resolve()

print(
    "Official PatchTST:",
    actual_model_file,
)

if (
    actual_model_file
    != expected_model_file
):
    raise RuntimeError(
        "Wrong PatchTST implementation imported.\n"
        f"Expected: {expected_model_file}\n"
        f"Actual:   {actual_model_file}"
    )

print(
    "PASS: official PatchTST implementation is active."
)


In [ ]:

DATA_PATHS = {
    "ETTh1": next(
        (
            p
            for p in [
                Path(
                    "/data/Time-Series-Library/"
                    "dataset/ETT-small/ETTh1.csv"
                ),
                Path(
                    "/data/dataset/ETTh1.csv"
                ),
            ]
            if p.is_file()
        ),
        None,
    ),
    "ETTm1": Path(
        "/data/dataset/ETTm1.csv"
    ),
}


def ett_unit(
    name,
):
    if name == "ETTh1":
        return (
            30
            * 24
        )

    if name == "ETTm1":
        return (
            30
            * 24
            * 4
        )

    raise ValueError(
        name
    )


def split_bounds(
    name,
):
    u = ett_unit(
        name
    )

    return (
        12 * u,
        16 * u,
        20 * u,
    )


def load_data(
    name,
):
    path = DATA_PATHS[
        name
    ]

    if (
        path is None
        or not path.is_file()
    ):
        raise FileNotFoundError(
            f"{name} dataset not found: {path}"
        )

    df = pd.read_csv(
        path
    )

    if "date" not in df.columns:
        raise ValueError(
            f"{name}: expected standard ETT CSV with a date column."
        )

    cols = [
        c
        for c in df.columns
        if c != "date"
    ]

    raw_all = df[
        cols
    ].to_numpy(
        dtype=np.float32
    )

    tr, va, te = split_bounds(
        name
    )

    if len(
        raw_all
    ) < te:
        raise ValueError(
            f"{name}: standard split needs {te} rows, "
            f"file contains {len(raw_all)}."
        )

    raw = raw_all[
        :te
    ].copy()

    if raw.shape[
        1
    ] != 7:
        raise ValueError(
            f"{name}: expected 7 channels, "
            f"found {raw.shape[1]}."
        )

    mu = raw[
        :tr
    ].mean(
        axis=0
    ).astype(
        np.float32
    )

    sd = raw[
        :tr
    ].std(
        axis=0
    ).astype(
        np.float32
    )

    if np.any(
        sd <= 1e-6
    ):
        raise ValueError(
            f"{name}: degenerate training channel."
        )

    z = (
        (
            raw
            - mu[
                None,
                :
            ]
        )
        / sd[
            None,
            :
        ]
    ).astype(
        np.float32
    )

    return {
        "name":
            name,
        "path":
            path,
        "columns":
            cols,
        "raw":
            raw,
        "z":
            z,
        "mean":
            mu,
        "std":
            sd,
        "n_channels":
            raw.shape[
                1
            ],
        "train_end":
            tr,
        "val_end":
            va,
        "test_end":
            te,
    }


DATA = {
    name:
        load_data(
            name
        )
    for name in DATASETS
}

display(
    pd.DataFrame([
        {
            "Dataset":
                name,
            "Path":
                str(
                    d[
                        "path"
                    ]
                ),
            "Channels":
                d[
                    "n_channels"
                ],
            "TrainEnd":
                d[
                    "train_end"
                ],
            "ValEnd":
                d[
                    "val_end"
                ],
            "TestEnd":
                d[
                    "test_end"
                ],
        }
        for name, d
        in DATA.items()
    ])
)


In [ ]:

def official_direct_ckpt_path(
    name,
):
    return (
        OFFICIAL_ROOT
        / "checkpoints"
        / (
            f"{name}_L336_H96_"
            "official_recipe_seed2021.pt"
        )
    )


def full_retriever_ckpt_path(
    name,
):
    return (
        FROZEN_RET_ROOT
        / "full_retriever"
        / f"{name}_H96_seed0.pt"
    )


def fold_retriever_ckpt_path(
    name,
    fold,
):
    return (
        FROZEN_RET_ROOT
        / "fold_retriever"
        / (
            f"{name}_H96_F{fold}_"
            "seed0.pt"
        )
    )


required = []

for name in DATASETS:
    required.append({
        "Dataset":
            name,
        "Kind":
            "Official full direct",
        "Path":
            official_direct_ckpt_path(
                name
            ),
    })

    required.append({
        "Dataset":
            name,
        "Kind":
            "Frozen full retriever",
        "Path":
            full_retriever_ckpt_path(
                name
            ),
    })

    for fold in range(
        1,
        4,
    ):
        required.append({
            "Dataset":
                name,
            "Kind":
                f"Frozen fold retriever F{fold}",
            "Path":
                fold_retriever_ckpt_path(
                    name,
                    fold,
                ),
        })

preflight = pd.DataFrame(
    required
)

preflight[
    "Exists"
] = preflight[
    "Path"
].map(
    lambda p:
        Path(
            p
        ).is_file()
)

display(
    preflight
)

missing = preflight[
    ~preflight[
        "Exists"
    ]
]

if len(
    missing
):
    print(
        "\nMissing prerequisites:"
    )

    for p in missing[
        "Path"
    ]:
        print(
            " -",
            p,
        )

    raise FileNotFoundError(
        "Required frozen checkpoints are missing. "
        "Experiment 15 and Experiment 13 H=96 must be completed first."
    )

print(
    "PASS: all official-direct and frozen-retriever checkpoints are available."
)


In [ ]:

def set_seed(
    seed,
):
    random.seed(
        seed
    )

    np.random.seed(
        seed
    )

    torch.manual_seed(
        seed
    )

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(
            seed
        )

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def load_torch(
    path,
):
    try:
        return torch.load(
            path,
            map_location=DEVICE,
            weights_only=False,
        )
    except TypeError:
        return torch.load(
            path,
            map_location=DEVICE,
        )


def ret_amp():
    if RETRIEVER_USE_AMP:
        return torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
        )

    return nullcontext()


def inv_softplus(
    x,
):
    return math.log(
        math.exp(
            float(
                x
            )
        )
        - 1.0
    )


class PredictivePatchEncoder(
    nn.Module
):
    def __init__(
        self,
    ):
        super().__init__()

        self.patch_proj = nn.Linear(
            REP_PATCH_LEN,
            REP_D_MODEL,
        )

        self.pos_embed = nn.Parameter(
            torch.zeros(
                1,
                REP_NUM_PATCHES,
                REP_D_MODEL,
            )
        )

        nn.init.trunc_normal_(
            self.pos_embed,
            std=0.02,
        )

        layer = nn.TransformerEncoderLayer(
            d_model=
                REP_D_MODEL,
            nhead=
                REP_N_HEADS,
            dim_feedforward=
                REP_D_FF,
            dropout=
                REP_DROPOUT,
            activation=
                "gelu",
            batch_first=
                True,
            norm_first=
                True,
        )

        self.encoder = nn.TransformerEncoder(
            layer,
            num_layers=
                REP_LAYERS,
        )

        self.norm = nn.LayerNorm(
            REP_D_MODEL
        )

        self.proj = nn.Linear(
            REP_D_MODEL,
            REP_DIM,
        )

    def forward(
        self,
        x,
    ):
        p = x.unfold(
            1,
            REP_PATCH_LEN,
            REP_PATCH_STRIDE,
        )

        h = (
            self.patch_proj(
                p
            )
            + self.pos_embed[
                :,
                :p.shape[
                    1
                ],
            ]
        )

        h = self.encoder(
            h
        ).mean(
            dim=1
        )

        h = self.proj(
            self.norm(
                h
            )
        )

        return F.normalize(
            h,
            dim=-1,
            eps=1e-8,
        )


class EmbeddingOnlyRetriever(
    nn.Module
):
    def __init__(
        self,
    ):
        super().__init__()

        self.encoder = (
            PredictivePatchEncoder()
        )

        self.raw_gamma = nn.Parameter(
            torch.tensor(
                inv_softplus(
                    1.0
                ),
                dtype=torch.float32,
            )
        )

    @property
    def gamma(
        self,
    ):
        return F.softplus(
            self.raw_gamma
        )

    def encode(
        self,
        x,
    ):
        return self.encoder(
            x
        )


class CrossFitAdaptiveGate(
    nn.Module
):
    def __init__(
        self,
    ):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(
                GATE_DIM,
                64,
            ),
            nn.LayerNorm(
                64
            ),
            nn.GELU(),
            nn.Dropout(
                0.1
            ),
            nn.Linear(
                64,
                32,
            ),
            nn.GELU(),
            nn.Dropout(
                0.1
            ),
            nn.Linear(
                32,
                1,
            ),
        )

        nn.init.normal_(
            self.net[
                -1
            ].weight,
            mean=0.0,
            std=1e-3,
        )

        nn.init.constant_(
            self.net[
                -1
            ].bias,
            math.log(
                0.1
                / 0.9
            ),
        )

    def forward(
        self,
        x,
    ):
        return torch.sigmoid(
            self.net(
                x
            ).squeeze(
                -1
            )
        )


In [ ]:

OFFICIAL_RECIPES = {
    "ETTh1": {
        "enc_in":
            7,
        "e_layers":
            3,
        "n_heads":
            4,
        "d_model":
            16,
        "d_ff":
            128,
        "dropout":
            0.3,
        "fc_dropout":
            0.3,
        "head_dropout":
            0.0,
        "patch_len":
            16,
        "stride":
            8,
        "batch_size":
            128,
        "learning_rate":
            1e-4,
        "train_epochs":
            100,
        "lradj":
            "type3",
        "pct_start":
            0.3,
    },
    "ETTm1": {
        "enc_in":
            7,
        "e_layers":
            3,
        "n_heads":
            16,
        "d_model":
            128,
        "d_ff":
            256,
        "dropout":
            0.2,
        "fc_dropout":
            0.2,
        "head_dropout":
            0.0,
        "patch_len":
            16,
        "stride":
            8,
        "batch_size":
            128,
        "learning_rate":
            1e-4,
        "train_epochs":
            100,
        "lradj":
            "TST",
        "pct_start":
            0.4,
    },
}


def official_config(
    name,
):
    r = OFFICIAL_RECIPES[
        name
    ]

    return SimpleNamespace(
        enc_in=
            r[
                "enc_in"
            ],
        seq_len=
            DIRECT_SEQ_LEN,
        pred_len=
            H,
        e_layers=
            r[
                "e_layers"
            ],
        n_heads=
            r[
                "n_heads"
            ],
        d_model=
            r[
                "d_model"
            ],
        d_ff=
            r[
                "d_ff"
            ],
        dropout=
            r[
                "dropout"
            ],
        fc_dropout=
            r[
                "fc_dropout"
            ],
        head_dropout=
            r[
                "head_dropout"
            ],
        individual=0,
        patch_len=
            r[
                "patch_len"
            ],
        stride=
            r[
                "stride"
            ],
        padding_patch=
            "end",
        revin=1,
        affine=0,
        subtract_last=0,
        decomposition=0,
        kernel_size=25,
    )


def build_official_model(
    name,
):
    return OfficialPatchTST(
        official_config(
            name
        )
    ).float().to(
        DEVICE
    )


def load_official_full_direct(
    name,
):
    path = official_direct_ckpt_path(
        name
    )

    ckpt = load_torch(
        path
    )

    model = build_official_model(
        name
    )

    model.load_state_dict(
        ckpt[
            "StateDict"
        ]
    )

    model.eval()

    return (
        model,
        ckpt,
    )


def load_frozen_retriever(
    path,
):
    ckpt = load_torch(
        path
    )

    model = EmbeddingOnlyRetriever().to(
        DEVICE
    )

    model.load_state_dict(
        ckpt[
            "StateDict"
        ]
    )

    model.eval()

    return (
        model,
        ckpt,
    )


meta_rows = []

for name in DATASETS:
    _, dck = load_official_full_direct(
        name
    )

    _, rck = load_frozen_retriever(
        full_retriever_ckpt_path(
            name
        )
    )

    meta_rows.append({
        "Dataset":
            name,
        "OfficialDirectBestEpoch":
            int(
                dck[
                    "BestEpoch"
                ]
            ),
        "OfficialDirectBestValMSE":
            float(
                dck[
                    "BestValMSE"
                ]
            ),
        "FrozenRetrieverBestEpoch":
            int(
                rck[
                    "BestEpoch"
                ]
            ),
        "FrozenRetrieverValAnalog":
            rck.get(
                "BestValAnalogFutureMSE",
                np.nan,
            ),
    })

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

meta_df = pd.DataFrame(
    meta_rows
)

display(
    meta_df
)


In [ ]:

def ret_anchors(
    start,
    end,
    stride=1,
):
    return np.arange(
        max(
            int(
                start
            ),
            RET_SEQ_LEN,
        ),
        int(
            end
        )
        - H
        + 1,
        int(
            stride
        ),
        dtype=np.int64,
    )


def eval_anchors(
    start,
    end,
    stride=1,
):
    # Strong direct model needs 336 historical points.
    return np.arange(
        max(
            int(
                start
            ),
            DIRECT_SEQ_LEN,
        ),
        int(
            end
        )
        - H
        + 1,
        int(
            stride
        ),
        dtype=np.int64,
    )


def batch_pattern(
    x,
):
    x = np.asarray(
        x,
        np.float32,
    )

    xc = (
        x
        - x.mean(
            axis=-1,
            keepdims=True,
        )
    )

    n = np.linalg.norm(
        xc,
        axis=-1,
        keepdims=True,
    )

    return np.where(
        n > EPS,
        xc
        / np.maximum(
            n,
            EPS,
        ),
        0.0,
    ).astype(
        np.float32
    )


def context7(
    x,
):
    x = np.asarray(
        x,
        np.float32,
    )

    short = max(
        8,
        RET_SEQ_LEN
        // 4,
    )

    m = x.mean(
        axis=-1
    )

    s = (
        x.std(
            axis=-1
        )
        + EPS
    )

    f1 = (
        x[
            ...,
            -1
        ]
        - m
    ) / s

    f2 = (
        x[
            ...,
            -short:
        ].mean(
            axis=-1
        )
        - m
    ) / s

    f3 = (
        x[
            ...,
            -1
        ]
        - x[
            ...,
            -short
        ]
    ) / s

    f4 = (
        x[
            ...,
            -1
        ]
        - x[
            ...,
            0
        ]
    ) / s

    df = np.diff(
        x,
        axis=-1,
    )

    ds = np.diff(
        x[
            ...,
            -short:
        ],
        axis=-1,
    )

    f5 = (
        ds.std(
            axis=-1
        )
        + EPS
    ) / (
        df.std(
            axis=-1
        )
        + EPS
    )

    t = np.linspace(
        -1.0,
        1.0,
        RET_SEQ_LEN,
        dtype=np.float32,
    )

    t = (
        t
        - t.mean()
    )

    f6 = (
        np.sum(
            t
            * (
                x
                - m[
                    ...,
                    None
                ]
            ),
            axis=-1,
        )
        / (
            np.sum(
                t
                * t
            )
            + EPS
        )
    ) / s

    a = x[
        ...,
        :-1
    ]

    b = x[
        ...,
        1:
    ]

    a = (
        a
        - a.mean(
            axis=-1,
            keepdims=True,
        )
    )

    b = (
        b
        - b.mean(
            axis=-1,
            keepdims=True,
        )
    )

    f7 = np.sum(
        a
        * b,
        axis=-1,
    ) / (
        np.sqrt(
            np.sum(
                a
                * a,
                axis=-1,
            )
            * np.sum(
                b
                * b,
                axis=-1,
            )
        )
        + EPS
    )

    return np.stack(
        [
            f1,
            f2,
            f3,
            f4,
            f5,
            f6,
            f7,
        ],
        axis=-1,
    ).astype(
        np.float32
    )


def extract_channel(
    z,
    c,
    aa,
):
    aa = np.asarray(
        aa,
        np.int64,
    )

    pi = (
        aa[
            :,
            None
        ]
        - RET_SEQ_LEN
        + np.arange(
            RET_SEQ_LEN
        )[
            None,
            :
        ]
    )

    fi = (
        aa[
            :,
            None
        ]
        + np.arange(
            H
        )[
            None,
            :
        ]
    )

    past = z[
        pi,
        c,
    ].astype(
        np.float32
    )

    future = z[
        fi,
        c,
    ].astype(
        np.float32
    )

    current = z[
        aa
        - 1,
        c,
    ].astype(
        np.float32
    )

    future_residual = (
        future
        - current[
            :,
            None
        ]
    ).astype(
        np.float32
    )

    return (
        past,
        future_residual,
    )


def build_memory(
    z,
    channels,
    boundary,
):
    memory_anchors = np.arange(
        RET_SEQ_LEN,
        int(
            boundary
        )
        - H
        + 1,
        MEMORY_STRIDE,
        dtype=np.int64,
    )

    if len(
        memory_anchors
    ) < TOP_K:
        raise ValueError(
            "Insufficient admissible retrieval memory."
        )

    past = np.empty(
        (
            channels,
            len(
                memory_anchors
            ),
            RET_SEQ_LEN,
        ),
        dtype=np.float32,
    )

    pattern = np.empty_like(
        past
    )

    future = np.empty(
        (
            channels,
            len(
                memory_anchors
            ),
            H,
        ),
        dtype=np.float32,
    )

    for c in range(
        channels
    ):
        p, f = extract_channel(
            z,
            c,
            memory_anchors,
        )

        past[
            c
        ] = p

        pattern[
            c
        ] = batch_pattern(
            p
        )

        future[
            c
        ] = f

    return {
        "anchors":
            memory_anchors,
        "past":
            past,
        "pattern":
            pattern,
        "future":
            future,
        "M":
            len(
                memory_anchors
            ),
        "boundary":
            int(
                boundary
            ),
    }


def prefix_norm(
    raw,
    prefix,
):
    mu = raw[
        :prefix
    ].mean(
        axis=0
    ).astype(
        np.float32
    )

    sd = raw[
        :prefix
    ].std(
        axis=0
    ).astype(
        np.float32
    )

    if np.any(
        sd <= 1e-6
    ):
        raise ValueError(
            "Degenerate prefix channel."
        )

    return (
        (
            raw
            - mu[
                None,
                :
            ]
        )
        / sd[
            None,
            :
        ]
    ).astype(
        np.float32
    )


In [ ]:

@torch.no_grad()
def encode_np(
    model,
    x,
    chunk=512,
):
    parts = []

    for i in range(
        0,
        len(
            x
        ),
        chunk,
    ):
        t = torch.from_numpy(
            x[
                i:
                i+chunk
            ]
        ).to(
            DEVICE
        )

        with ret_amp():
            e = model.encode(
                t
            ).float()

        parts.append(
            e
        )

        del t, e

    return torch.cat(
        parts,
        dim=0,
    )


@torch.no_grad()
def memory_gpu(
    model,
    memory,
    channels,
):
    emb = torch.empty(
        (
            channels,
            memory[
                "M"
            ],
            REP_DIM,
        ),
        dtype=torch.float32,
        device=DEVICE,
    )

    for c in range(
        channels
    ):
        emb[
            c
        ] = encode_np(
            model,
            memory[
                "past"
            ][
                c
            ],
        )

    return {
        "emb":
            emb,
        "pattern":
            torch.from_numpy(
                memory[
                    "pattern"
                ]
            ).to(
                DEVICE
            ),
        "future":
            torch.from_numpy(
                memory[
                    "future"
                ]
            ).to(
                DEVICE
            ),
    }


def query_pairs(
    z,
    aa,
    cc,
):
    aa = np.asarray(
        aa,
        np.int64,
    )

    cc = np.asarray(
        cc,
        np.int64,
    )

    pi = (
        aa[
            :,
            None
        ]
        - RET_SEQ_LEN
        + np.arange(
            RET_SEQ_LEN
        )[
            None,
            :
        ]
    )

    fi = (
        aa[
            :,
            None
        ]
        + np.arange(
            H
        )[
            None,
            :
        ]
    )

    past = z[
        pi,
        cc[
            :,
            None
        ],
    ].astype(
        np.float32
    )

    future = z[
        fi,
        cc[
            :,
            None
        ],
    ].astype(
        np.float32
    )

    current = z[
        aa
        - 1,
        cc,
    ].astype(
        np.float32
    )

    true_residual = (
        future
        - current[
            :,
            None
        ]
    ).astype(
        np.float32
    )

    return (
        past,
        batch_pattern(
            past
        ),
        context7(
            past
        ),
        true_residual,
    )


@torch.no_grad()
def retrieve(
    model,
    memory_gpu_obj,
    z,
    aa,
    cc,
):
    past, pattern, ctx, true = query_pairs(
        z,
        aa,
        cc,
    )

    past_t = torch.from_numpy(
        past
    ).to(
        DEVICE
    )

    pattern_t = torch.from_numpy(
        pattern
    ).to(
        DEVICE
    )

    with ret_amp():
        qemb = model.encode(
            past_t
        )

    qemb = qemb.float()

    memb = memory_gpu_obj[
        "emb"
    ][
        cc
    ]

    sim = torch.bmm(
        qemb[
            :,
            None,
            :
        ],
        memb.transpose(
            1,
            2,
        ),
    ).squeeze(
        1
    )

    score = (
        model.gamma
        * sim
    )

    idx = torch.topk(
        score,
        TOP_K,
        dim=1,
    ).indices

    row = torch.arange(
        len(
            cc
        ),
        device=DEVICE,
    )[
        :,
        None
    ]

    pfull = torch.bmm(
        pattern_t[
            :,
            None,
            :
        ],
        memory_gpu_obj[
            "pattern"
        ][
            cc
        ].transpose(
            1,
            2,
        ),
    ).squeeze(
        1
    )

    return {
        "score":
            score[
                row,
                idx
            ],
        "sim":
            sim[
                row,
                idx
            ],
        "pattern":
            pfull[
                row,
                idx
            ],
        "cand":
            memory_gpu_obj[
                "future"
            ][
                cc[
                    :,
                    None
                ],
                idx,
            ],
        "ctx":
            torch.from_numpy(
                ctx
            ).to(
                DEVICE
            ),
        "true":
            torch.from_numpy(
                true
            ).to(
                DEVICE
            ),
    }


In [ ]:

@torch.no_grad()
def direct_residual(
    model,
    z,
    aa,
):
    aa = np.asarray(
        aa,
        np.int64,
    )

    pi = (
        aa[
            :,
            None
        ]
        - DIRECT_SEQ_LEN
        + np.arange(
            DIRECT_SEQ_LEN
        )[
            None,
            :
        ]
    )

    x = torch.from_numpy(
        z[
            pi,
            :
        ].astype(
            np.float32
        )
    ).to(
        DEVICE
    )

    pred = model(
        x
    ).float()

    # Convert absolute normalized forecast to residual forecast.
    residual = (
        pred
        - x[
            :,
            -1:,
            :
        ].float()
    )

    del x, pred

    return residual


@torch.no_grad()
def direct_full_test_metric(
    model,
    data,
):
    aa = eval_anchors(
        data[
            "val_end"
        ],
        data[
            "test_end"
        ],
        stride=1,
    )

    sse = 0.0
    sae = 0.0
    n = 0

    for i in range(
        0,
        len(
            aa
        ),
        FULL_ANCHOR_BATCH,
    ):
        a = aa[
            i:
            i+FULL_ANCHOR_BATCH
        ]

        d3 = direct_residual(
            model,
            data[
                "z"
            ],
            a,
        )

        A = len(
            a
        )

        C = data[
            "n_channels"
        ]

        pa = np.repeat(
            a,
            C,
        )

        pc = np.tile(
            np.arange(
                C,
                dtype=np.int64,
            ),
            A,
        )

        _, _, _, true = query_pairs(
            data[
                "z"
            ],
            pa,
            pc,
        )

        d = d3.permute(
            0,
            2,
            1,
        ).reshape(
            -1,
            H,
        )

        y = torch.from_numpy(
            true
        ).to(
            DEVICE
        )

        e = (
            d
            - y
        )

        sse += float(
            (
                e
                * e
            ).sum()
        )

        sae += float(
            e.abs().sum()
        )

        n += e.numel()

        del (
            d3,
            d,
            y,
            e,
        )

    return (
        sse
        / n,
        sae
        / n,
        len(
            aa
        ),
    )


In [ ]:

EXP15_RESULTS = (
    OFFICIAL_ROOT
    / "official_patchtst_h96_results.csv"
)

exp15 = (
    pd.read_csv(
        EXP15_RESULTS
    )
    if EXP15_RESULTS.exists()
    else pd.DataFrame()
)

parity_rows = []

for name in DATASETS:
    model, ckpt = load_official_full_direct(
        name
    )

    mse, mae, n_windows = direct_full_test_metric(
        model,
        DATA[
            name
        ],
    )

    expected_mse = np.nan
    expected_mae = np.nan

    if len(
        exp15
    ):
        hit = exp15[
            exp15[
                "Dataset"
            ]
            == name
        ]

        if len(
            hit
        ):
            expected_mse = float(
                hit.iloc[
                    0
                ][
                    "FullStride1_MSE"
                ]
            )

            expected_mae = float(
                hit.iloc[
                    0
                ][
                    "FullStride1_MAE"
                ]
            )

    parity_rows.append({
        "Dataset":
            name,
        "CurrentMSE":
            mse,
        "Exp15MSE":
            expected_mse,
        "AbsMSEDiff":
            abs(
                mse
                - expected_mse
            )
            if np.isfinite(
                expected_mse
            )
            else np.nan,
        "CurrentMAE":
            mae,
        "Exp15MAE":
            expected_mae,
        "TestWindows":
            n_windows,
        "BestEpoch":
            int(
                ckpt[
                    "BestEpoch"
                ]
            ),
    })

    del model

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

parity_df = pd.DataFrame(
    parity_rows
)

display(
    parity_df
)

if (
    parity_df[
        "AbsMSEDiff"
    ].notna().all()
    and (
        parity_df[
            "AbsMSEDiff"
        ]
        < 2e-5
    ).all()
):
    print(
        "PASS: direct evaluation matches Experiment 15 FullStride1."
    )
else:
    print(
        "WARNING: inspect direct parity before trusting augmentation results."
    )


In [ ]:

def fold_direct_path(
    name,
    fold,
):
    return (
        DIRS[
            "fold_direct"
        ]
        / (
            f"{name}_H96_F{fold}_"
            "official_direct.pt"
        )
    )


def adjust_type3_lr(
    optimizer,
    base_lr,
    epoch,
):
    lr = (
        base_lr
        if epoch < 3
        else (
            base_lr
            * (
                0.9
                ** (
                    epoch
                    - 3
                )
            )
        )
    )

    for group in optimizer.param_groups:
        group[
            "lr"
        ] = lr

    return lr


def direct_train_anchors(
    end,
):
    return np.arange(
        DIRECT_SEQ_LEN,
        int(
            end
        )
        - H
        + 1,
        dtype=np.int64,
    )


def make_direct_batch(
    z,
    aa,
):
    aa = np.asarray(
        aa,
        np.int64,
    )

    pi = (
        aa[
            :,
            None
        ]
        - DIRECT_SEQ_LEN
        + np.arange(
            DIRECT_SEQ_LEN
        )[
            None,
            :
        ]
    )

    fi = (
        aa[
            :,
            None
        ]
        + np.arange(
            H
        )[
            None,
            :
        ]
    )

    return (
        z[
            pi,
            :
        ].astype(
            np.float32
        ),
        z[
            fi,
            :
        ].astype(
            np.float32
        ),
    )


def train_fold_official_direct(
    name,
    z,
    prefix,
    fixed_epochs,
    fold,
):
    path = fold_direct_path(
        name,
        fold,
    )

    if (
        path.exists()
        and RESUME
        and not FORCE
    ):
        ckpt = load_torch(
            path
        )

        model = build_official_model(
            name
        )

        model.load_state_dict(
            ckpt[
                "StateDict"
            ]
        )

        model.eval()

        print(
            f"Loaded fold direct: {path.name}"
        )

        return (
            model,
            ckpt,
        )

    r = OFFICIAL_RECIPES[
        name
    ]

    seed = (
        CROSSFIT_SEED
        + H
        * 100
        + int(
            prefix
        )
        % 997
    )

    set_seed(
        seed
    )

    model = build_official_model(
        name
    )

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=
            r[
                "learning_rate"
            ],
    )

    train_aa = direct_train_anchors(
        prefix
    )

    steps_per_epoch = (
        len(
            train_aa
        )
        // r[
            "batch_size"
        ]
    )

    if steps_per_epoch < 1:
        raise ValueError(
            f"{name} fold {fold}: insufficient training windows."
        )

    onecycle = torch.optim.lr_scheduler.OneCycleLR(
        optimizer=
            optimizer,
        steps_per_epoch=
            steps_per_epoch,
        pct_start=
            r[
                "pct_start"
            ],
        epochs=
            r[
                "train_epochs"
            ],
        max_lr=
            r[
                "learning_rate"
            ],
    )

    rng = np.random.default_rng(
        seed
        + 1
    )

    hist = []

    for epoch in range(
        1,
        int(
            fixed_epochs
        )
        + 1,
    ):
        model.train()

        order = rng.permutation(
            len(
                train_aa
            )
        )

        # Match official drop_last=True behavior.
        usable = (
            len(
                order
            )
            // r[
                "batch_size"
            ]
        ) * r[
            "batch_size"
        ]

        order = order[
            :usable
        ]

        losses = []

        for left in range(
            0,
            usable,
            r[
                "batch_size"
            ],
        ):
            ids = order[
                left:
                left
                + r[
                    "batch_size"
                ]
            ]

            x_np, y_np = make_direct_batch(
                z,
                train_aa[
                    ids
                ],
            )

            x = torch.from_numpy(
                x_np
            ).to(
                DEVICE
            )

            y = torch.from_numpy(
                y_np
            ).to(
                DEVICE
            )

            optimizer.zero_grad(
                set_to_none=True
            )

            pred = model(
                x
            )

            loss = F.mse_loss(
                pred,
                y,
            )

            loss.backward()

            optimizer.step()

            losses.append(
                float(
                    loss.item()
                )
            )

            if r[
                "lradj"
            ] == "TST":
                current = onecycle.get_last_lr()[
                    0
                ]

                for group in optimizer.param_groups:
                    group[
                        "lr"
                    ] = current

                onecycle.step()

            del (
                x,
                y,
                pred,
                loss,
            )

        if r[
            "lradj"
        ] != "TST":
            lr = adjust_type3_lr(
                optimizer,
                r[
                    "learning_rate"
                ],
                epoch,
            )
        else:
            lr = onecycle.get_last_lr()[
                0
            ]

        train_mse = float(
            np.mean(
                losses
            )
        )

        hist.append({
            "Epoch":
                epoch,
            "TrainMSE":
                train_mse,
            "LR":
                lr,
        })

        print(
            f"Fold direct {name:6s} "
            f"F{fold} "
            f"ep={epoch:03d}/{fixed_epochs} "
            f"train={train_mse:.6f} "
            f"lr={lr:.3e}"
        )

    model.eval()

    state = {
        k:
            v.detach()
            .cpu()
            .clone()
        for k, v
        in model.state_dict().items()
    }

    ckpt = {
        "Dataset":
            name,
        "Fold":
            fold,
        "Prefix":
            int(
                prefix
            ),
        "FixedEpochs":
            int(
                fixed_epochs
            ),
        "Seed":
            seed,
        "StateDict":
            state,
    }

    torch.save(
        ckpt,
        path,
    )

    pd.DataFrame(
        hist
    ).to_csv(
        DIRS[
            "history"
        ]
        / (
            f"{path.stem}_history.csv"
        ),
        index=False,
    )

    return (
        model,
        ckpt,
    )


In [ ]:

def score_entropy(
    s,
):
    p = torch.softmax(
        s,
        dim=1,
    )

    return (
        -(
            p
            * torch.log(
                p.clamp_min(
                    1e-8
                )
            )
        ).sum(
            dim=1
        )
        / math.log(
            TOP_K
        )
    )


def feature_cosine(
    a,
    b,
):
    return (
        (
            a
            * b
        ).sum(
            dim=1
        )
        / (
            torch.sqrt(
                (
                    a
                    * a
                ).sum(
                    dim=1
                )
                + 1e-8
            )
            * torch.sqrt(
                (
                    b
                    * b
                ).sum(
                    dim=1
                )
                + 1e-8
            )
        )
    )


def gate_features(
    r,
    retrieval,
    direct,
):
    s = r[
        "score"
    ]

    sim = r[
        "sim"
    ]

    pattern = r[
        "pattern"
    ]

    sorted_s = torch.sort(
        s,
        dim=1,
        descending=True,
    ).values

    cand_std = r[
        "cand"
    ].std(
        dim=1,
        unbiased=False,
    )

    disp_rms = torch.sqrt(
        (
            cand_std
            * cand_std
        ).mean(
            dim=1
        )
        + 1e-8
    )

    disp_mean = cand_std.mean(
        dim=1
    )

    direct_rms = torch.sqrt(
        (
            direct
            * direct
        ).mean(
            dim=1
        )
        + 1e-8
    )

    retrieval_rms = torch.sqrt(
        (
            retrieval
            * retrieval
        ).mean(
            dim=1
        )
        + 1e-8
    )

    disagreement = (
        retrieval
        - direct
    )

    disagreement_rms = torch.sqrt(
        (
            disagreement
            * disagreement
        ).mean(
            dim=1
        )
        + 1e-8
    )

    relative_disagreement = (
        disagreement_rms
        / (
            direct_rms
            + retrieval_rms
            + 1e-6
        )
    )

    scalars = torch.stack(
        [
            s.mean(
                dim=1
            ),
            s.std(
                dim=1,
                unbiased=False,
            ),
            s.max(
                dim=1
            ).values,
            sorted_s[
                :,
                0
            ]
            - sorted_s[
                :,
                1
            ],
            s.max(
                dim=1
            ).values
            - s.mean(
                dim=1
            ),
            score_entropy(
                s
            ),
            sim.mean(
                dim=1
            ),
            sim.std(
                dim=1,
                unbiased=False,
            ),
            sim.max(
                dim=1
            ).values,
            pattern.mean(
                dim=1
            ),
            pattern.std(
                dim=1,
                unbiased=False,
            ),
            pattern.max(
                dim=1
            ).values,
            disp_rms,
            disp_mean,
            direct_rms,
            retrieval_rms,
            disagreement_rms,
            relative_disagreement,
            feature_cosine(
                direct,
                retrieval,
            ),
        ],
        dim=1,
    )

    out = torch.cat(
        [
            r[
                "ctx"
            ],
            scalars,
        ],
        dim=1,
    )

    if out.shape[
        1
    ] != GATE_DIM:
        raise RuntimeError(
            f"Gate feature dimension mismatch: {out.shape}"
        )

    return out


def abc_terms(
    direct,
    retrieval,
    true,
):
    e = (
        direct
        - true
    )

    delta = (
        retrieval
        - direct
    )

    return torch.stack(
        [
            (
                e
                * e
            ).mean(
                dim=1
            ),
            (
                e
                * delta
            ).mean(
                dim=1
            ),
            (
                delta
                * delta
            ).mean(
                dim=1
            ),
        ],
        dim=1,
    )


In [ ]:

@torch.no_grad()
def collect_gate_data(
    data,
    direct_model,
    retriever,
    memory_gpu_obj,
    z,
    aa,
):
    C = data[
        "n_channels"
    ]

    features = []
    abcs = []
    anchors_out = []
    channels_out = []

    for i in range(
        0,
        len(
            aa
        ),
        FULL_ANCHOR_BATCH,
    ):
        a = aa[
            i:
            i+FULL_ANCHOR_BATCH
        ]

        A = len(
            a
        )

        d3 = direct_residual(
            direct_model,
            z,
            a,
        )

        pair_anchor = np.repeat(
            a,
            C,
        )

        pair_channel = np.tile(
            np.arange(
                C,
                dtype=np.int64,
            ),
            A,
        )

        r = retrieve(
            retriever,
            memory_gpu_obj,
            z,
            pair_anchor,
            pair_channel,
        )

        retrieval = r[
            "cand"
        ].mean(
            dim=1
        )

        direct = d3.permute(
            0,
            2,
            1,
        ).reshape(
            -1,
            H,
        )

        true = r[
            "true"
        ]

        f = gate_features(
            r,
            retrieval,
            direct,
        )

        a_terms = abc_terms(
            direct,
            retrieval,
            true,
        )

        features.append(
            f.cpu()
            .numpy()
            .astype(
                np.float32
            )
        )

        abcs.append(
            a_terms.cpu()
            .numpy()
            .astype(
                np.float32
            )
        )

        anchors_out.append(
            pair_anchor
        )

        channels_out.append(
            pair_channel
        )

        del (
            d3,
            r,
            retrieval,
            direct,
            true,
            f,
            a_terms,
        )

    return {
        "feature":
            np.concatenate(
                features,
                axis=0,
            ),
        "abc":
            np.concatenate(
                abcs,
                axis=0,
            ),
        "anchor":
            np.concatenate(
                anchors_out,
                axis=0,
            ),
        "channel":
            np.concatenate(
                channels_out,
                axis=0,
            ),
    }


def oof_path(
    name,
    fold,
):
    return (
        DIRS[
            "oof"
        ]
        / (
            f"{name}_H96_F{fold}_"
            "official_direct.npz"
        )
    )


def build_oof_fold(
    data,
    fold,
    p0,
    p1,
    direct_epochs,
):
    name = data[
        "name"
    ]

    C = data[
        "n_channels"
    ]

    out_path = oof_path(
        name,
        fold,
    )

    if (
        out_path.exists()
        and RESUME
        and not FORCE
    ):
        obj = np.load(
            out_path
        )

        print(
            f"Loaded OOF: {out_path.name}"
        )

        return {
            key:
                obj[
                    key
                ]
            for key in [
                "feature",
                "abc",
                "anchor",
                "channel",
            ]
        }

    prefix = int(
        p0
        * data[
            "train_end"
        ]
    )

    oof_end = int(
        p1
        * data[
            "train_end"
        ]
    )

    z = prefix_norm(
        data[
            "raw"
        ],
        prefix,
    )

    direct_model, _ = train_fold_official_direct(
        name,
        z,
        prefix,
        direct_epochs,
        fold,
    )

    retriever, _ = load_frozen_retriever(
        fold_retriever_ckpt_path(
            name,
            fold,
        )
    )

    memory = build_memory(
        z,
        C,
        prefix,
    )

    memory_gpu_obj = memory_gpu(
        retriever,
        memory,
        C,
    )

    aa = eval_anchors(
        prefix,
        oof_end,
        stride=
            OOF_ANCHOR_STRIDE,
    )

    print(
        f"OOF {name} F{fold}: "
        f"prefix={prefix}, end={oof_end}, "
        f"anchors={len(aa)}, pairs={len(aa)*C}, "
        f"memory/C={memory['M']}"
    )

    out = collect_gate_data(
        data,
        direct_model,
        retriever,
        memory_gpu_obj,
        z,
        aa,
    )

    np.savez_compressed(
        out_path,
        **out,
    )

    del (
        direct_model,
        retriever,
        memory,
        memory_gpu_obj,
        z,
    )

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return out


In [ ]:

def fit_feature_scaler(
    x,
):
    median = np.median(
        x,
        axis=0,
    ).astype(
        np.float32
    )

    q25 = np.percentile(
        x,
        25,
        axis=0,
    )

    q75 = np.percentile(
        x,
        75,
        axis=0,
    )

    iqr = (
        q75
        - q25
    ).astype(
        np.float32
    )

    iqr = np.where(
        iqr < 1e-5,
        1.0,
        iqr,
    ).astype(
        np.float32
    )

    return (
        median,
        iqr,
    )


def scale_features(
    x,
    median,
    iqr,
):
    return np.clip(
        (
            x
            - median
        )
        / iqr,
        -8.0,
        8.0,
    ).astype(
        np.float32
    )


def gate_loss(
    alpha,
    abc,
):
    return (
        abc[
            :,
            0
        ]
        + 2.0
        * alpha
        * abc[
            :,
            1
        ]
        + alpha
        * alpha
        * abc[
            :,
            2
        ]
    ).mean()


def gate_checkpoint_path(
    name,
):
    return (
        DIRS[
            "gate"
        ]
        / f"{name}_H96_official_direct.pt"
    )


def train_gate_epoch(
    model,
    optimizer,
    x,
    abc,
    rng,
):
    model.train()

    order = rng.permutation(
        len(
            x
        )
    )

    losses = []

    for i in range(
        0,
        len(
            order
        ),
        GATE_BATCH,
    ):
        ids = order[
            i:
            i+GATE_BATCH
        ]

        xt = torch.from_numpy(
            x[
                ids
            ]
        ).to(
            DEVICE
        )

        at = torch.from_numpy(
            abc[
                ids
            ]
        ).to(
            DEVICE
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        alpha = model(
            xt
        )

        loss = gate_loss(
            alpha,
            at,
        )

        loss.backward()

        optimizer.step()

        losses.append(
            float(
                loss.item()
            )
        )

        del (
            xt,
            at,
            alpha,
            loss,
        )

    return float(
        np.mean(
            losses
        )
    )


@torch.no_grad()
def evaluate_gate(
    model,
    x,
    abc,
):
    model.eval()

    total = 0.0
    n = 0
    alpha_sum = 0.0

    for i in range(
        0,
        len(
            x
        ),
        GATE_BATCH,
    ):
        xt = torch.from_numpy(
            x[
                i:
                i+GATE_BATCH
            ]
        ).to(
            DEVICE
        )

        at = torch.from_numpy(
            abc[
                i:
                i+GATE_BATCH
            ]
        ).to(
            DEVICE
        )

        alpha = model(
            xt
        )

        each = (
            at[
                :,
                0
            ]
            + 2.0
            * alpha
            * at[
                :,
                1
            ]
            + alpha
            * alpha
            * at[
                :,
                2
            ]
        )

        total += float(
            each.sum()
        )

        n += len(
            alpha
        )

        alpha_sum += float(
            alpha.sum()
        )

        del (
            xt,
            at,
            alpha,
            each,
        )

    return (
        total
        / n,
        alpha_sum
        / n,
    )


def train_crossfit_gate(
    name,
    oof_x,
    oof_abc,
    val_x,
    val_abc,
):
    path = gate_checkpoint_path(
        name
    )

    if (
        path.exists()
        and RESUME
        and not FORCE
    ):
        ckpt = load_torch(
            path
        )

        model = CrossFitAdaptiveGate().to(
            DEVICE
        )

        model.load_state_dict(
            ckpt[
                "StateDict"
            ]
        )

        model.eval()

        print(
            f"Loaded gate: {path.name}"
        )

        return (
            model,
            ckpt,
        )

    median, iqr = fit_feature_scaler(
        oof_x
    )

    train_x = scale_features(
        oof_x,
        median,
        iqr,
    )

    valid_x = scale_features(
        val_x,
        median,
        iqr,
    )

    seed = (
        CROSSFIT_SEED
        + H
        * 3000
        + sum(
            map(
                ord,
                name,
            )
        )
    )

    set_seed(
        seed
    )

    model = CrossFitAdaptiveGate().to(
        DEVICE
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=GATE_LR,
        weight_decay=GATE_WD,
    )

    rng = np.random.default_rng(
        seed
        + 1
    )

    best = float(
        "inf"
    )

    best_epoch = -1
    wait = 0
    hist = []

    for epoch in range(
        1,
        GATE_MAX_EPOCHS
        + 1,
    ):
        train_mse = train_gate_epoch(
            model,
            optimizer,
            train_x,
            oof_abc,
            rng,
        )

        val_mse, mean_alpha = evaluate_gate(
            model,
            valid_x,
            val_abc,
        )

        hist.append({
            "Epoch":
                epoch,
            "OOFTrainMSE":
                train_mse,
            "ValMSE":
                val_mse,
            "ValMeanAlpha":
                mean_alpha,
        })

        if (
            val_mse
            < best
            - 1e-10
        ):
            best = val_mse
            best_epoch = epoch
            wait = 0
        else:
            wait += 1

        print(
            f"Gate {name:6s} "
            f"ep={epoch:02d} "
            f"OOF={train_mse:.6f} "
            f"val={val_mse:.6f} "
            f"alpha={mean_alpha:.3f} "
            f"best={best:.6f}@{best_epoch}"
        )

        if (
            wait
            >= GATE_PATIENCE
        ):
            break

    # Reinitialize and train ONLY on OOF data
    # for the validation-selected number of epochs.
    set_seed(
        seed
    )

    final = CrossFitAdaptiveGate().to(
        DEVICE
    )

    final_opt = torch.optim.AdamW(
        final.parameters(),
        lr=GATE_LR,
        weight_decay=GATE_WD,
    )

    final_rng = np.random.default_rng(
        seed
        + 2
    )

    for _ in range(
        best_epoch
    ):
        train_gate_epoch(
            final,
            final_opt,
            train_x,
            oof_abc,
            final_rng,
        )

    final.eval()

    ckpt = {
        "BestEpoch":
            best_epoch,
        "BestValMSE":
            best,
        "FeatureMedian":
            median,
        "FeatureIQR":
            iqr,
        "StateDict": {
            k:
                v.detach()
                .cpu()
                .clone()
            for k, v
            in final.state_dict().items()
        },
    }

    torch.save(
        ckpt,
        path,
    )

    pd.DataFrame(
        hist
    ).to_csv(
        DIRS[
            "history"
        ]
        / f"{path.stem}_history.csv",
        index=False,
    )

    return (
        final,
        ckpt,
    )


def mse_scalar(
    abc,
    alpha,
):
    A = abc.astype(
        np.float64
    )

    x = float(
        alpha
    )

    return float(
        np.mean(
            A[
                :,
                0
            ]
            + 2.0
            * x
            * A[
                :,
                1
            ]
            + x
            * x
            * A[
                :,
                2
            ]
        )
    )


def choose_scalar(
    abc,
):
    rows = []

    best_alpha = None
    best_mse = float(
        "inf"
    )

    for alpha in ALPHA_GRID:
        mse = mse_scalar(
            abc,
            alpha,
        )

        rows.append({
            "Alpha":
                float(
                    alpha
                ),
            "MSE":
                mse,
        })

        if (
            mse
            < best_mse
            - 1e-10
        ):
            best_mse = mse
            best_alpha = float(
                alpha
            )

    return (
        best_alpha,
        pd.DataFrame(
            rows
        ),
    )


@torch.no_grad()
def gate_alpha(
    model,
    ckpt,
    x,
):
    sx = scale_features(
        x,
        ckpt[
            "FeatureMedian"
        ],
        ckpt[
            "FeatureIQR"
        ],
    )

    outputs = []

    for i in range(
        0,
        len(
            sx
        ),
        GATE_BATCH,
    ):
        t = torch.from_numpy(
            sx[
                i:
                i+GATE_BATCH
            ]
        ).to(
            DEVICE
        )

        outputs.append(
            model(
                t
            ).cpu()
            .numpy()
        )

        del t

    return np.concatenate(
        outputs
    ).astype(
        np.float32
    )


def mse_pair(
    abc,
    alpha,
):
    A = abc.astype(
        np.float64
    )

    x = np.asarray(
        alpha,
        dtype=np.float64,
    )

    return float(
        np.mean(
            A[
                :,
                0
            ]
            + 2.0
            * x
            * A[
                :,
                1
            ]
            + x
            * x
            * A[
                :,
                2
            ]
        )
    )


def choose_lambda(
    abc,
    gate_alpha_values,
    scalar_alpha,
):
    rows = []

    best_lambda = None
    best_mse = float(
        "inf"
    )

    for lmb in LAMBDA_GRID:
        alpha = (
            (
                1.0
                - float(
                    lmb
                )
            )
            * scalar_alpha
            + float(
                lmb
            )
            * gate_alpha_values
        )

        mse = mse_pair(
            abc,
            alpha,
        )

        rows.append({
            "Lambda":
                float(
                    lmb
                ),
            "MSE":
                mse,
            "MeanAlpha":
                float(
                    alpha.mean()
                ),
        })

        if (
            mse
            < best_mse
            - 1e-10
        ):
            best_mse = mse
            best_lambda = float(
                lmb
            )

    return (
        best_lambda,
        pd.DataFrame(
            rows
        ),
    )


In [ ]:

def empty_stat():
    return {
        "sse":
            0.0,
        "sae":
            0.0,
        "n":
            0,
    }


def update_stat(
    stat,
    pred,
    true,
):
    e = (
        pred
        - true
    )

    stat[
        "sse"
    ] += float(
        (
            e
            * e
        ).sum()
    )

    stat[
        "sae"
    ] += float(
        e.abs().sum()
    )

    stat[
        "n"
    ] += e.numel()


def finish_stat(
    stat,
):
    return (
        stat[
            "sse"
        ]
        / stat[
            "n"
        ],
        stat[
            "sae"
        ]
        / stat[
            "n"
        ],
    )


def oracle_prediction(
    direct,
    retrieval,
    true,
):
    e = (
        direct
        - true
    )

    delta = (
        retrieval
        - direct
    )

    alpha = torch.clamp(
        -(
            e
            * delta
        ).sum(
            dim=1
        )
        / (
            (
                delta
                * delta
            ).sum(
                dim=1
            )
            + 1e-8
        ),
        0.0,
        1.0,
    )

    return (
        direct
        + alpha[
            :,
            None
        ]
        * delta
    )


@torch.no_grad()
def test_evaluate(
    data,
    direct_model,
    retriever,
    memory_gpu_obj,
    gate,
    gate_ckpt,
    scalar_alpha,
    shrink_lambda,
):
    C = data[
        "n_channels"
    ]

    aa = eval_anchors(
        data[
            "val_end"
        ],
        data[
            "test_end"
        ],
        stride=1,
    )

    stats = {
        key:
            empty_stat()
        for key in [
            "Direct",
            "Retrieval",
            "Scalar",
            "RawAdaptive",
            "ShrinkAdaptive",
            "Oracle",
        ]
    }

    anchor_mse = {
        key:
            []
        for key in [
            "Direct",
            "Retrieval",
            "Scalar",
            "RawAdaptive",
            "ShrinkAdaptive",
            "Oracle",
        ]
    }

    raw_alpha_sum = 0.0
    shrink_alpha_sum = 0.0
    n_pairs = 0

    for i in range(
        0,
        len(
            aa
        ),
        FULL_ANCHOR_BATCH,
    ):
        a = aa[
            i:
            i+FULL_ANCHOR_BATCH
        ]

        A = len(
            a
        )

        d3 = direct_residual(
            direct_model,
            data[
                "z"
            ],
            a,
        )

        pair_anchor = np.repeat(
            a,
            C,
        )

        pair_channel = np.tile(
            np.arange(
                C,
                dtype=np.int64,
            ),
            A,
        )

        r = retrieve(
            retriever,
            memory_gpu_obj,
            data[
                "z"
            ],
            pair_anchor,
            pair_channel,
        )

        retrieval = r[
            "cand"
        ].mean(
            dim=1
        )

        direct = d3.permute(
            0,
            2,
            1,
        ).reshape(
            -1,
            H,
        )

        true = r[
            "true"
        ]

        scalar = (
            direct
            + scalar_alpha
            * (
                retrieval
                - direct
            )
        )

        features = gate_features(
            r,
            retrieval,
            direct,
        ).cpu().numpy().astype(
            np.float32
        )

        scaled = scale_features(
            features,
            gate_ckpt[
                "FeatureMedian"
            ],
            gate_ckpt[
                "FeatureIQR"
            ],
        )

        gate_alpha_values = gate(
            torch.from_numpy(
                scaled
            ).to(
                DEVICE
            )
        )

        shrink_alpha_values = (
            (
                1.0
                - shrink_lambda
            )
            * scalar_alpha
            + shrink_lambda
            * gate_alpha_values
        )

        raw_adaptive = (
            direct
            + gate_alpha_values[
                :,
                None
            ]
            * (
                retrieval
                - direct
            )
        )

        shrink_adaptive = (
            direct
            + shrink_alpha_values[
                :,
                None
            ]
            * (
                retrieval
                - direct
            )
        )

        oracle = oracle_prediction(
            direct,
            retrieval,
            true,
        )

        predictions = {
            "Direct":
                direct,
            "Retrieval":
                retrieval,
            "Scalar":
                scalar,
            "RawAdaptive":
                raw_adaptive,
            "ShrinkAdaptive":
                shrink_adaptive,
            "Oracle":
                oracle,
        }

        for key, pred in predictions.items():
            update_stat(
                stats[
                    key
                ],
                pred,
                true,
            )

            per_pair = (
                (
                    pred
                    - true
                )
                ** 2
            ).mean(
                dim=1
            ).reshape(
                A,
                C,
            ).mean(
                dim=1
            )

            anchor_mse[
                key
            ].extend(
                per_pair.cpu()
                .numpy()
                .tolist()
            )

        raw_alpha_sum += float(
            gate_alpha_values.sum()
        )

        shrink_alpha_sum += float(
            shrink_alpha_values.sum()
        )

        n_pairs += len(
            gate_alpha_values
        )

        del (
            d3,
            r,
            retrieval,
            direct,
            true,
            scalar,
            features,
            scaled,
            gate_alpha_values,
            shrink_alpha_values,
            raw_adaptive,
            shrink_adaptive,
            oracle,
            predictions,
        )

    metrics = {
        key:
            finish_stat(
                value
            )
        for key, value
        in stats.items()
    }

    return {
        "anchors":
            aa,
        "metrics":
            metrics,
        "anchor_mse": {
            key:
                np.asarray(
                    value,
                    dtype=np.float32,
                )
            for key, value
            in anchor_mse.items()
        },
        "raw_mean_alpha":
            raw_alpha_sum
            / n_pairs,
        "shrink_mean_alpha":
            shrink_alpha_sum
            / n_pairs,
    }


In [ ]:

def moving_block_bootstrap(
    difference,
    n_boot=5000,
    block=24,
    seed=131313,
):
    x = np.asarray(
        difference,
        dtype=np.float64,
    )

    n = len(
        x
    )

    L = min(
        block,
        n,
    )

    rng = np.random.default_rng(
        seed
    )

    n_blocks = int(
        np.ceil(
            n
            / L
        )
    )

    max_start = max(
        1,
        n
        - L
        + 1,
    )

    boot = np.empty(
        n_boot,
        dtype=np.float64,
    )

    for b in range(
        n_boot
    ):
        starts = rng.integers(
            0,
            max_start,
            size=n_blocks,
        )

        sample = np.concatenate(
            [
                x[
                    s:
                    s+L
                ]
                for s in starts
            ]
        )[
            :n
        ]

        boot[
            b
        ] = sample.mean()

    return {
        "MeanImprovement":
            float(
                x.mean()
            ),
        "CI_Low":
            float(
                np.quantile(
                    boot,
                    0.025,
                )
            ),
        "CI_High":
            float(
                np.quantile(
                    boot,
                    0.975,
                )
            ),
    }


In [ ]:

SUMMARY_PATH = (
    ROOT
    / "summary.csv"
)

BOOTSTRAP_PATH = (
    ROOT
    / "bootstrap.csv"
)

CALIBRATION_PATH = (
    ROOT
    / "calibration.csv"
)

FOLD_PATH = (
    ROOT
    / "folds.csv"
)

existing = (
    pd.read_csv(
        SUMMARY_PATH
    )
    if (
        RESUME
        and SUMMARY_PATH.exists()
    )
    else pd.DataFrame()
)

summary_rows = (
    existing.to_dict(
        "records"
    )
    if len(
        existing
    )
    else []
)

bootstrap_rows = []
calibration_frames = []
fold_rows = []


def already_done(
    name,
):
    if not len(
        existing
    ):
        return False

    return bool(
        (
            existing[
                "Dataset"
            ]
            == name
        ).any()
    )


for name in DATASETS:
    if already_done(
        name
    ):
        print(
            f"SKIP completed dataset: {name}"
        )
        continue

    start_time = time.time()

    data = DATA[
        name
    ]

    C = data[
        "n_channels"
    ]

    print(
        "\n"
        + "#"
        * 130
    )

    print(
        f"{name} | OFFICIAL PATCHTST + FROZEN RETRIEVAL | H=96"
    )

    print(
        "#"
        * 130
    )

    # --------------------------------------------------------
    # 1. Load frozen full upstream components.
    # --------------------------------------------------------
    direct_model, direct_ckpt = load_official_full_direct(
        name
    )

    retriever, retriever_ckpt = load_frozen_retriever(
        full_retriever_ckpt_path(
            name
        )
    )

    direct_epochs = int(
        direct_ckpt[
            "BestEpoch"
        ]
    )

    print(
        "Frozen full direct best epoch:",
        direct_epochs,
    )

    print(
        "Frozen retriever best epoch:",
        retriever_ckpt[
            "BestEpoch"
        ],
    )

    # --------------------------------------------------------
    # 2. Build chronological OOF data.
    # --------------------------------------------------------
    oof_parts = []

    for fold, (
        p0,
        p1,
    ) in enumerate(
        FOLDS,
        start=1,
    ):
        part = build_oof_fold(
            data,
            fold,
            p0,
            p1,
            direct_epochs,
        )

        oof_parts.append(
            part
        )

        fold_rows.append({
            "Dataset":
                name,
            "Fold":
                fold,
            "PrefixFrac":
                p0,
            "OOFEndFrac":
                p1,
            "Pairs":
                len(
                    part[
                        "feature"
                    ]
                ),
            "Anchors":
                len(
                    np.unique(
                        part[
                            "anchor"
                        ]
                    )
                ),
            "DirectFixedEpochs":
                direct_epochs,
            "RetrieverCheckpoint":
                str(
                    fold_retriever_ckpt_path(
                        name,
                        fold,
                    )
                ),
        })

    oof_x = np.concatenate(
        [
            part[
                "feature"
            ]
            for part in oof_parts
        ],
        axis=0,
    )

    oof_abc = np.concatenate(
        [
            part[
                "abc"
            ]
            for part in oof_parts
        ],
        axis=0,
    )

    print(
        "Total OOF pairs:",
        len(
            oof_x
        ),
    )

    # --------------------------------------------------------
    # 3. Validation:
    #    full official direct +
    #    frozen full retriever +
    #    train-only retrieval memory.
    # --------------------------------------------------------
    val_memory = build_memory(
        data[
            "z"
        ],
        C,
        data[
            "train_end"
        ],
    )

    val_memory_gpu = memory_gpu(
        retriever,
        val_memory,
        C,
    )

    val_aa = eval_anchors(
        data[
            "train_end"
        ],
        data[
            "val_end"
        ],
        stride=1,
    )

    val_data = collect_gate_data(
        data,
        direct_model,
        retriever,
        val_memory_gpu,
        data[
            "z"
        ],
        val_aa,
    )

    gate, gate_ckpt = train_crossfit_gate(
        name,
        oof_x,
        oof_abc,
        val_data[
            "feature"
        ],
        val_data[
            "abc"
        ],
    )

    scalar_alpha, scalar_curve = choose_scalar(
        val_data[
            "abc"
        ]
    )

    raw_val_alpha = gate_alpha(
        gate,
        gate_ckpt,
        val_data[
            "feature"
        ],
    )

    shrink_lambda, lambda_curve = choose_lambda(
        val_data[
            "abc"
        ],
        raw_val_alpha,
        scalar_alpha,
    )

    scalar_curve[
        "Dataset"
    ] = name

    scalar_curve[
        "Kind"
    ] = "ScalarAlpha"

    lambda_curve[
        "Dataset"
    ] = name

    lambda_curve[
        "Kind"
    ] = "ShrinkLambda"

    lambda_curve[
        "ScalarAlpha"
    ] = scalar_alpha

    calibration_frames.extend(
        [
            scalar_curve,
            lambda_curve,
        ]
    )

    print(
        f"Validation calibration | "
        f"alpha0={scalar_alpha:.2f} | "
        f"lambda={shrink_lambda:.2f} | "
        f"gateEpoch={gate_ckpt['BestEpoch']}"
    )

    del (
        val_memory,
        val_memory_gpu,
        val_data,
        oof_x,
        oof_abc,
        oof_parts,
        raw_val_alpha,
    )

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # --------------------------------------------------------
    # 4. Static test:
    #    direct model remains train-only;
    #    retrieval memory may use train+val histories only.
    # --------------------------------------------------------
    test_memory = build_memory(
        data[
            "z"
        ],
        C,
        data[
            "val_end"
        ],
    )

    test_memory_gpu = memory_gpu(
        retriever,
        test_memory,
        C,
    )

    test = test_evaluate(
        data,
        direct_model,
        retriever,
        test_memory_gpu,
        gate,
        gate_ckpt,
        scalar_alpha,
        shrink_lambda,
    )

    metrics = test[
        "metrics"
    ]

    direct_mse, direct_mae = metrics[
        "Direct"
    ]

    retrieval_mse, retrieval_mae = metrics[
        "Retrieval"
    ]

    scalar_mse, scalar_mae = metrics[
        "Scalar"
    ]

    raw_mse, raw_mae = metrics[
        "RawAdaptive"
    ]

    shrink_mse, shrink_mae = metrics[
        "ShrinkAdaptive"
    ]

    oracle_mse, oracle_mae = metrics[
        "Oracle"
    ]

    # --------------------------------------------------------
    # 5. Paired bootstrap.
    # Positive difference means the second method is better.
    # --------------------------------------------------------
    b_direct = moving_block_bootstrap(
        test[
            "anchor_mse"
        ][
            "Direct"
        ]
        - test[
            "anchor_mse"
        ][
            "ShrinkAdaptive"
        ],
        seed=
            131313
            + sum(
                map(
                    ord,
                    name,
                )
            ),
    )

    bootstrap_rows.append({
        "Dataset":
            name,
        "Comparison":
            "Direct-ShrinkAdaptive",
        **b_direct,
        "SignificantPositive":
            b_direct[
                "CI_Low"
            ]
            > 0.0,
    })

    b_scalar = moving_block_bootstrap(
        test[
            "anchor_mse"
        ][
            "Scalar"
        ]
        - test[
            "anchor_mse"
        ][
            "ShrinkAdaptive"
        ],
        seed=
            232323
            + sum(
                map(
                    ord,
                    name,
                )
            ),
    )

    bootstrap_rows.append({
        "Dataset":
            name,
        "Comparison":
            "Scalar-ShrinkAdaptive",
        **b_scalar,
        "SignificantPositive":
            b_scalar[
                "CI_Low"
            ]
            > 0.0,
    })

    # --------------------------------------------------------
    # 6. Direct parity reference from Experiment 15.
    # --------------------------------------------------------
    exp15_mse = np.nan

    if len(
        exp15
    ):
        hit = exp15[
            exp15[
                "Dataset"
            ]
            == name
        ]

        if len(
            hit
        ):
            exp15_mse = float(
                hit.iloc[
                    0
                ][
                    "FullStride1_MSE"
                ]
            )

    row = {
        "Dataset":
            name,
        "Horizon":
            H,
        "Channels":
            C,
        "OfficialPatchTST_MSE":
            direct_mse,
        "OfficialPatchTST_MAE":
            direct_mae,
        "Exp15ReferenceMSE":
            exp15_mse,
        "DirectParityAbsDiff":
            abs(
                direct_mse
                - exp15_mse
            )
            if np.isfinite(
                exp15_mse
            )
            else np.nan,
        "Retrieval_MSE":
            retrieval_mse,
        "Retrieval_MAE":
            retrieval_mae,
        "ScalarAlpha":
            scalar_alpha,
        "Scalar_MSE":
            scalar_mse,
        "Scalar_MAE":
            scalar_mae,
        "RawAdaptive_MSE":
            raw_mse,
        "RawAdaptive_MAE":
            raw_mae,
        "ShrinkLambda":
            shrink_lambda,
        "ShrinkAdaptive_MSE":
            shrink_mse,
        "ShrinkAdaptive_MAE":
            shrink_mae,
        "Oracle_MSE":
            oracle_mse,
        "Oracle_MAE":
            oracle_mae,
        "RawMeanAlpha":
            test[
                "raw_mean_alpha"
            ],
        "ShrinkMeanAlpha":
            test[
                "shrink_mean_alpha"
            ],
        "ScalarGainVsDirect_pct":
            100.0
            * (
                direct_mse
                - scalar_mse
            )
            / direct_mse,
        "ShrinkGainVsDirect_pct":
            100.0
            * (
                direct_mse
                - shrink_mse
            )
            / direct_mse,
        "ShrinkGainVsScalar_pct":
            100.0
            * (
                scalar_mse
                - shrink_mse
            )
            / scalar_mse,
        "OracleHeadroomFromDirect_pct":
            100.0
            * (
                direct_mse
                - oracle_mse
            )
            / direct_mse,
        "OracleHeadroomFromShrink_pct":
            100.0
            * (
                shrink_mse
                - oracle_mse
            )
            / shrink_mse,
        "OfficialDirectBestEpoch":
            direct_epochs,
        "FrozenRetrieverBestEpoch":
            int(
                retriever_ckpt[
                    "BestEpoch"
                ]
            ),
        "GateBestEpoch":
            int(
                gate_ckpt[
                    "BestEpoch"
                ]
            ),
        "TestMemoryPerChannel":
            int(
                test_memory[
                    "M"
                ]
            ),
        "TestWindows":
            len(
                test[
                    "anchors"
                ]
            ),
        "RuntimeMinutes":
            (
                time.time()
                - start_time
            )
            / 60.0,
    }

    summary_rows.append(
        row
    )

    np.savez_compressed(
        DIRS[
            "paired"
        ]
        / f"{name}_H96_anchor_mse.npz",
        TestAnchors=
            test[
                "anchors"
            ],
        Direct=
            test[
                "anchor_mse"
            ][
                "Direct"
            ],
        Retrieval=
            test[
                "anchor_mse"
            ][
                "Retrieval"
            ],
        Scalar=
            test[
                "anchor_mse"
            ][
                "Scalar"
            ],
        RawAdaptive=
            test[
                "anchor_mse"
            ][
                "RawAdaptive"
            ],
        ShrinkAdaptive=
            test[
                "anchor_mse"
            ][
                "ShrinkAdaptive"
            ],
        Oracle=
            test[
                "anchor_mse"
            ][
                "Oracle"
            ],
    )

    pd.DataFrame(
        summary_rows
    ).to_csv(
        SUMMARY_PATH,
        index=False,
    )

    pd.DataFrame(
        bootstrap_rows
    ).to_csv(
        BOOTSTRAP_PATH,
        index=False,
    )

    pd.DataFrame(
        fold_rows
    ).to_csv(
        FOLD_PATH,
        index=False,
    )

    if calibration_frames:
        pd.concat(
            calibration_frames,
            ignore_index=True,
        ).to_csv(
            CALIBRATION_PATH,
            index=False,
        )

    display(
        pd.DataFrame([
            row
        ])[
            [
                "Dataset",
                "OfficialPatchTST_MSE",
                "Retrieval_MSE",
                "Scalar_MSE",
                "RawAdaptive_MSE",
                "ShrinkAdaptive_MSE",
                "Oracle_MSE",
                "ScalarAlpha",
                "ShrinkLambda",
                "ShrinkGainVsDirect_pct",
                "ShrinkGainVsScalar_pct",
                "ShrinkMeanAlpha",
            ]
        ]
    )

    display(
        pd.DataFrame(
            bootstrap_rows
        )[
            pd.DataFrame(
                bootstrap_rows
            )[
                "Dataset"
            ]
            == name
        ]
    )

    del (
        direct_model,
        direct_ckpt,
        retriever,
        retriever_ckpt,
        gate,
        gate_ckpt,
        test_memory,
        test_memory_gpu,
        test,
    )

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


summary_df = pd.DataFrame(
    summary_rows
).sort_values(
    [
        "Dataset",
        "Horizon",
    ]
).reset_index(
    drop=True
)

display(
    summary_df
)


In [ ]:

if not len(
    summary_df
):
    raise RuntimeError(
        "No completed results."
    )

compact = summary_df[
    [
        "Dataset",
        "OfficialPatchTST_MSE",
        "Scalar_MSE",
        "RawAdaptive_MSE",
        "ShrinkAdaptive_MSE",
        "Oracle_MSE",
        "ScalarAlpha",
        "ShrinkLambda",
        "ShrinkGainVsDirect_pct",
        "ShrinkMeanAlpha",
    ]
].copy()

compact[
    "Winner"
] = np.where(
    compact[
        "ShrinkAdaptive_MSE"
    ]
    < compact[
        "OfficialPatchTST_MSE"
    ],
    "Ours",
    "Direct",
)

display(
    compact
)

if BOOTSTRAP_PATH.exists():
    boot_df = pd.read_csv(
        BOOTSTRAP_PATH
    )

    display(
        boot_df[
            boot_df[
                "Comparison"
            ]
            == "Direct-ShrinkAdaptive"
        ][
            [
                "Dataset",
                "MeanImprovement",
                "CI_Low",
                "CI_High",
                "SignificantPositive",
            ]
        ]
    )


In [ ]:

print(
    "Experiment root:",
    ROOT,
)

for p in sorted(
    ROOT.rglob(
        "*"
    )
):
    if p.is_file():
        print(
            p.relative_to(
                ROOT
            )
        )
